# Mamba-130m — autoLRP attribution

State-space LM from `state-spaces/mamba-130m-hf`. The selective scan inside `MambaMixer.slow_forward` is just a Python loop of standard ops (`mul`, `matmul`, `exp`, `softplus`), so autoLRP's existing rules cover it without any model-specific code. We force the slow path by running on CPU.

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import torch
from transformers import AutoTokenizer, MambaForCausalLM

import autolrp
from autolrp import LRPConfig
from _common import show_text_attribution, show_text_comparison

# CPU forces the slow_forward decomposition path so the scan is visible to autograd.
device = 'cpu'
MODEL_ID = 'state-spaces/mamba-130m-hf'

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = MambaForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).eval().to(device)

class MambaFromEmb(torch.nn.Module):
    """Embeddings → next-token logits (mean-centered for LRP)."""
    def __init__(self, m):
        super().__init__()
        self.body = m.backbone
        self.head = m.lm_head
    def forward(self, inputs_embeds):
        h = self.body(inputs_embeds=inputs_embeds).last_hidden_state
        logits = self.head(h)[:, -1, :]
        return logits

wrapper = MambaFromEmb(model).eval().to(device)

In [ ]:
text = 'The capital of France is'
ids = tok(text, return_tensors='pt').input_ids.to(device)
emb = model.backbone.embeddings(ids).detach()

with torch.no_grad():
    pred = wrapper(emb).argmax(-1).item()
print(f'Top next token: {tok.decode([pred])!r}')

In [ ]:
x = autolrp.tensor(emb)
out = wrapper(x)
out[0, pred].lrp()                          # BASE; Mamba has no attention, so no attn= preset applies

tokens = tok.convert_ids_to_tokens(ids[0])
rel = x.relevance[0].sum(-1).detach().cpu().numpy()
show_text_attribution(tokens, rel, predicted_token=tok.decode([pred]), prompt_text='Mamba: state-space attribution on factual recall')